In [ ]:
# !pip install langchain
# !pip install langchain-community
# !pip install pypdf

In [ ]:
csv_data = "https://raw.githubusercontent.com/Shikher-jain/LLM-Course/refs/heads/main/DataCamp/csv_data.csv"
html_data = "https://raw.githubusercontent.com/Shikher-jain/LLM-Course/refs/heads/main/DataCamp/html_data.html"
pdf_data = "https://raw.githubusercontent.com/Shikher-jain/LLM-Course/refs/heads/main/DataCamp/pdf_data.pdf"

In [ ]:
import pprint

In [ ]:
# Import library
from langchain_community.document_loaders import PyPDFLoader, UnstructuredHTMLLoader

# Create a document loader for rag_paper.pdf
loader = PyPDFLoader('pdf_data.pdf')

# Load the document
pdf_data = loader.load()

print(pdf_data[0].page_content)
print(pdf_data[0].metadata)

In [ ]:
# !pip install unstructured

In [ ]:
# Import library
from langchain_community.document_loaders import UnstructuredHTMLLoader

# Load the document
loader= UnstructuredHTMLLoader("html_data.html")
html_data = loader.load()

# Print the first document's content
print(html_data[0].page_content)

# Print the first document's metadata
print(html_data[0].metadata)

In [ ]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader("csv_data.csv")
csv_data = loader.load()

print(csv_data[0].page_content)
print(csv_data[-1].metadata)

In [ ]:
# !pip install langchain_text_splitters

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    separator='.',
    chunk_size=75,
    chunk_overlap=10
)
# Split the text using text_splitter
chunks = text_splitter.split_text(csv_data[0].page_content)
pprint.pprint(chunks)
pprint.pprint([len(chunk) for chunk in chunks])

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    separators=['.'],
    chunk_size=75,
    chunk_overlap=10
)

# Split the text using text_splitter
chunks = text_splitter.split_text(csv_data[-1].page_content)
# chunks = text_splitter.split_documents(pdf_data)
pprint.pprint(chunks)
print([len(chunk) for chunk in chunks])

In [ ]:
# !pip install langchain_chroma langchain_openai

In [ ]:
# !pip install langchain_community
# !pip install sentence-transformers

In [ ]:
# !pip install -U chromadb langchain langchain-community langchain-core
# !pip install -U opentelemetry-api opentelemetry-sdk

In [ ]:
# Downgrade opentelemetry-sdk and opentelemetry-api to a compatible version
!pip install opentelemetry-sdk==1.19.0 opentelemetry-api==1.19.0
# After running this cell, please restart your runtime by going to Runtime > Restart runtime.

In [ ]:
# Re-running the cell after runtime restart to verify the fix.
# from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# Create an instance of the OpenAIEmbeddings class

# embedding_model = OpenAIEmbeddings(
    # api_key="<OPENAI_API_TOKEN>",
    #   model='text-embedding-3-small')

from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create a Chroma vector store and embed the chunks
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)


In [ ]:
# from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# Create an instance of the OpenAIEmbeddings class

# embedding_model = OpenAIEmbeddings(
    # api_key="<OPENAI_API_TOKEN>",
    #   model='text-embedding-3-small')

from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create a Chroma vector store and embed the chunks
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)

In [ ]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 2},
    search_type="similarity",
    # search_type="mmr",
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# prompt = ChatPromptTemplate.from_messages()
prompt = ChatPromptTemplate.from_template(
    "You are a helpful assistant. Answer the question based on the context provided.\n\nContext: {context}\n\nQuestion: {question}\n\nAnswer:"
)

In [ ]:
# llm = ChatOpenAI(model="gpt-3.5-turbo",api_key="your-api-key", temperature=0)

from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

pipe = pipeline(
    "text-generation",
    model="mistralai/Mistral-7B-Instruct-v0.2",
    device=0  # GPU (use -1 for CPU)
)

llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [ ]:
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


In [ ]:
result = chain.invoke({"question": "What is the main topic of the document?"})
print(result)

In [ ]:
prompt = """
Use the only the context provided to answer the following question. If you don't know the answer, reply that you are unsure.
Context: {context}
Question: {question}
"""

from langchain_core.prompts import ChatPromptTemplate


# Convert the string into a chat prompt template
prompt_template = ChatPromptTemplate.from_template(prompt)

# Create an LCEL chain to test the prompt
chain = prompt_template | llm

# Invoke the chain on the inputs provided
print(chain.invoke({"context": "DataCamp's RAG course was created by Meri Nova and James Chapman!", "question": "Who created DataCamp's RAG course?"}))

In [ ]:
# Convert the vector store into a retriever
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k":2})

# Create the LCEL retrieval chain
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

# Invoke the chain
print(chain.invoke("Who are the authors?"))